# 面试问题：怎样从零实现可靠的 JSON Structured Output？

可直接复述的回答：只在生成结束后解析 JSON 太晚，模型已经可能走进无效前缀。可靠方案把有限 schema 编译成 token 或字符状态机，每步只保留仍可到达合法终态的候选。语法约束只保证 JSON 形状，订单归属、枚举和金额范围仍需语义校验。动态字段不能全量枚举，应在请求作用域内构造允许值或使用语法状态机。约束失败要有明确 fallback，而不是静默改写。Schema、tokenizer 和解码器版本必须一起发布。

后续代码使用可读业务记录验证回答主线。实验结果均标注为教学实验，只用于解释机制，不代表真实线上收益。


## 1. 真实案例：取消订单动作与输入预览

五条脱敏取消请求包含订单号和原因。工具 schema 要求 `action=cancel_order`，订单号来自当前用户可见集合，原因只能是三个枚举值。


In [1]:
import json  # 使用标准库展示真实 JSON 文本。
requests07 = [  # 构造五条订单取消请求。
    {"request_id": "r1", "order_id": "o-201", "reason": "duplicate"},  # 重复下单。
    {"request_id": "r2", "order_id": "o-202", "reason": "changed_mind"},  # 用户改变主意。
    {"request_id": "r3", "order_id": "o-203", "reason": "wrong_address"},  # 地址填写错误。
    {"request_id": "r4", "order_id": "o-204", "reason": "duplicate"},  # 第二条重复下单请求。
    {"request_id": "r5", "order_id": "o-205", "reason": "changed_mind"},  # 第二条改变主意请求。
]  # 完成具有真实字段的结构化动作样本。
allowed_reasons07 = {"duplicate", "changed_mind", "wrong_address"}  # 冻结 schema 允许的原因枚举。
print("教学实验输入：取消订单请求")  # 标识输入预览。
for request07 in requests07:  # 逐条展示请求作用域字段。
    print(request07)  # 输出一条取消请求。


教学实验输入：取消订单请求
{'request_id': 'r1', 'order_id': 'o-201', 'reason': 'duplicate'}
{'request_id': 'r2', 'order_id': 'o-202', 'reason': 'changed_mind'}
{'request_id': 'r3', 'order_id': 'o-203', 'reason': 'wrong_address'}
{'request_id': 'r4', 'order_id': 'o-204', 'reason': 'duplicate'}
{'request_id': 'r5', 'order_id': 'o-205', 'reason': 'changed_mind'}


## 2. Baseline（基线）：贪心选择最高分完整字符串

模拟模型给出三个候选，其中最高分文本缺少引号并包含非法原因。后验 `json.loads` 只能在生成完成后报错，无法避免浪费和重试。


In [2]:
candidates07 = [  # 构造模型可能产生的完整候选。
    (0.92, '{"action":"cancel_order","order_id":"o-201","reason":duplicate}'),  # 最高分候选不是合法 JSON。
    (0.88, '{"action":"cancel_order","order_id":"o-201","reason":"duplicate"}'),  # 次高分候选合法且语义正确。
    (0.81, '{"action":"cancel_order","order_id":"o-201","reason":"other"}'),  # 语法合法但枚举非法。
]  # 完成生成候选列表。
baseline_text07 = max(candidates07, key=lambda item07: item07[0])[1]  # 贪心选择模型分数最高的字符串。
try:  # 捕获后验解析错误用于教学展示。
    baseline_object07 = json.loads(baseline_text07)  # 尝试在生成结束后解析 JSON。
    baseline_error07 = None  # 记录意外解析成功。
except json.JSONDecodeError as error07:  # 处理无效 JSON 文本。
    baseline_object07 = None  # 标记没有得到结构化对象。
    baseline_error07 = f"{error07.msg}@{error07.pos}"  # 保存稳定且小型的错误位置。
print("基线最高分文本", baseline_text07)  # 展示贪心解码结果。
print("后验解析结果", baseline_object07, "error=", baseline_error07)  # 展示生成结束后才发现错误。


基线最高分文本 {"action":"cancel_order","order_id":"o-201","reason":duplicate}
后验解析结果 None error= Expecting value@53


## 3. 核心实现：字符 Trie 与逐步 allowed-next

教学版把请求作用域内的有限合法 JSON 编译成字符 Trie。每个前缀只允许 Trie 子节点中的字符；下面输出关键位置的 allowed-next，直观看到非法未加引号原因被屏蔽。


In [3]:
class TrieNode07:  # 定义字符约束 Trie 节点。
    def __init__(self):  # 初始化子边和终态标记。
        self.children = {}  # 保存字符到子节点的映射。
        self.terminal = False  # 标记当前前缀是否为完整合法输出。
def build_trie07(texts07):  # 把有限合法文本编译成 Trie。
    root07 = TrieNode07()  # 创建空前缀根节点。
    for text07 in texts07:  # 逐个插入合法 JSON 字符串。
        node07 = root07  # 从根节点开始遍历字符。
        for character07 in text07:  # 逐字符扩展合法前缀。
            node07 = node07.children.setdefault(character07, TrieNode07())  # 创建或复用下一字符节点。
        node07.terminal = True  # 标记完整字符串终态。
    return root07  # 返回编译完成的约束 Trie。
def accepts07(root07, text07):  # 检查完整文本是否沿合法前缀到达终态。
    node07 = root07  # 从空前缀开始检查。
    for character07 in text07:  # 按生成顺序遍历字符。
        if character07 not in node07.children:  # 检查当前字符是否被约束允许。
            return False  # 非法前缀立即拒绝。
        node07 = node07.children[character07]  # 转移到下一约束状态。
    return node07.terminal  # 只有完整终态才接受输出。
valid_texts07 = [json.dumps({"action": "cancel_order", "order_id": "o-201", "reason": reason07}, separators=(",", ":"), sort_keys=True) for reason07 in sorted(allowed_reasons07)]  # 编译当前订单与原因枚举的合法文本。
trie07 = build_trie07(valid_texts07)  # 构建请求作用域字符 Trie。
constrained_candidates07 = [item07 for item07 in candidates07 if accepts07(trie07, item07[1])]  # 屏蔽不能到达合法终态的候选。
constrained_text07 = max(constrained_candidates07, key=lambda item07: item07[0])[1]  # 在合法候选中保留模型最高分。
trace_prefixes07 = ["", constrained_text07[:12], constrained_text07[:36], constrained_text07[:-1]]  # 选择四个关键前缀观察状态。
print("约束解码前缀：prefix -> allowed next")  # 输出 Trie 状态轨迹表头。
for prefix07 in trace_prefixes07:  # 逐个查询关键前缀允许字符。
    node07 = trie07  # 从根节点重放当前前缀。
    for character07 in prefix07:  # 沿前缀推进 Trie 状态。
        node07 = node07.children[character07]  # 转移到确定子节点。
    print(repr(prefix07), "->", sorted(node07.children))  # 展示下一步允许字符集合。
print("约束解码结果", constrained_text07)  # 展示合法最高分结构化输出。


约束解码前缀：prefix -> allowed next
'' -> ['{']
'{"action":"c' -> ['a']
'{"action":"cancel_order","order_id":' -> ['"']
'{"action":"cancel_order","order_id":"o-201","reason":"duplicate"' -> ['}']
约束解码结果 {"action":"cancel_order","order_id":"o-201","reason":"duplicate"}


## 4. 结果表与结果解读

约束解码拒绝了最高分无效 JSON，也拒绝了语法合法但枚举非法的候选。Trie 在这里同时编码语法和有限枚举；生产中通常把 JSON grammar 与业务 validator 分开。


In [4]:
def semantic_validate07(text07, allowed_orders07):  # 对解析后的对象执行业务语义校验。
    object07 = json.loads(text07)  # 将已通过语法约束的文本解析为对象。
    fields_ok07 = set(object07) == {"action", "order_id", "reason"}  # 检查字段集合严格匹配 schema。
    action_ok07 = object07["action"] == "cancel_order"  # 检查动作枚举。
    order_ok07 = object07["order_id"] in allowed_orders07  # 检查订单在当前主体作用域内。
    reason_ok07 = object07["reason"] in allowed_reasons07  # 检查原因枚举。
    return fields_ok07 and action_ok07 and order_ok07 and reason_ok07  # 返回组合语义校验结果。
valid07 = semantic_validate07(constrained_text07, {"o-201"})  # 校验最终文本的订单归属和枚举。
print("方法 | JSON可解析 | schema语义 | 结果")  # 输出基线和约束方案对照表头。
print("后验贪心", baseline_object07 is not None, False, "retry")  # 展示贪心输出无法解析。
print("Trie约束", True, valid07, "accepted")  # 展示约束输出通过语法和语义校验。
print("结果解读：逐前缀约束减少无效生成，语义 validator 仍不可省略")  # 解释语法约束与业务校验的分工。


方法 | JSON可解析 | schema语义 | 结果
后验贪心 False False retry
Trie约束 True True accepted
结果解读：逐前缀约束减少无效生成，语义 validator 仍不可省略


## 5. 失败案例与修正：静态 Trie 不认识新订单

若把所有订单号在发布时静态枚举，新创建的合法订单会被拒绝。修正是在请求进入时从服务端授权上下文取得可见订单，并动态编译有限值；不能让模型自行扩充 allowed list。


In [5]:
new_request07 = {"request_id": "r6", "order_id": "o-299", "reason": "duplicate"}  # 构造发布后新增的合法订单。
new_text07 = json.dumps({"action": "cancel_order", "order_id": new_request07["order_id"], "reason": new_request07["reason"]}, separators=(",", ":"), sort_keys=True)  # 生成新请求的目标 JSON。
static_accept07 = accepts07(trie07, new_text07)  # 演示旧请求作用域 Trie 拒绝新订单。
dynamic_texts07 = [json.dumps({"action": "cancel_order", "order_id": new_request07["order_id"], "reason": reason07}, separators=(",", ":"), sort_keys=True) for reason07 in sorted(allowed_reasons07)]  # 使用服务端授权订单动态生成合法空间。
dynamic_trie07 = build_trie07(dynamic_texts07)  # 为新请求编译作用域 Trie。
dynamic_accept07 = accepts07(dynamic_trie07, new_text07)  # 检查动态约束接受合法新订单。
print("失败行为：静态Trie接受新订单", static_accept07)  # 展示过度枚举导致的误拒绝。
print("修正行为：请求作用域Trie接受新订单", dynamic_accept07)  # 展示服务端动态值修正。


失败行为：静态Trie接受新订单 False
修正行为：请求作用域Trie接受新订单 True


## 6. 生产边界与解码制品

字符 Trie 便于教学但真实模型按 tokenizer token 解码；Unicode、数字和任意字符串需要 JSON grammar/DFA。Schema validator 还要限制长度、数值和资源权限，并提供超时与 fallback。


In [6]:
decoder_contract07 = {"schema": "cancel-order-v3", "tokenizer": "model-tokenizer-sha256", "decoder": "json-dfa-v2", "dynamic_values": "server_authorized", "fallback": "clarify_or_reject"}  # 定义结构化解码发布合同。
print("结构化解码制品", decoder_contract07)  # 展示 schema、tokenizer 和状态机必须一起版本化。
print("生产替换点：token级DFA、Unicode与数字状态、长度预算、资源授权和失败监控")  # 说明字符 Trie 的教学边界。


结构化解码制品 {'schema': 'cancel-order-v3', 'tokenizer': 'model-tokenizer-sha256', 'decoder': 'json-dfa-v2', 'dynamic_values': 'server_authorized', 'fallback': 'clarify_or_reject'}
生产替换点：token级DFA、Unicode与数字状态、长度预算、资源授权和失败监控


## 7. 最小回归测试

断言保护无效基线、合法约束输出和动态值修正。


In [7]:
assert len(requests07) >= 5  # 保证案例包含足够多的结构化请求。
assert baseline_object07 is None  # 保证后验解析失败案例仍可复现。
assert accepts07(trie07, constrained_text07)  # 保证约束输出到达合法终态。
assert valid07 is True  # 保证最终对象通过业务语义校验。
assert static_accept07 is False and dynamic_accept07 is True  # 保证动态授权值修正静态枚举缺陷。
print("最小回归测试通过：语法约束、语义校验和动态值作用域稳定")  # 显示结构化输出关键性质已验证。


最小回归测试通过：语法约束、语义校验和动态值作用域稳定
